In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error

from uni2ts.model.moirai import MoiraiForecast
from uni2ts.model.moirai.module import MoiraiModule

In [ ]:
df = pd.read_csv("../data/processed/monthly_labor_market.csv")

df.rename(columns={"date": "ds"}, inplace=True)
df["ds"] = pd.to_datetime(df["ds"], format="%Y:%m")

df = df.dropna(how="all", subset=df.columns[1:])
df = df.ffill()
df = df.dropna()

TARGET = "EMPLOY"

split = int(len(df) * 0.8)

train_df = df.iloc[:split]
test_df = df.iloc[split:split+12]

y_true = test_df[TARGET].values

In [ ]:
# Historical context: full series with major economic events and train/test boundary
EVENTS = {
    "2001-09-01": "9/11 / Dot-com bust",
    "2008-09-01": "Financial crisis",
    "2020-03-01": "COVID-19",
}

y_max = df[TARGET].max()
y_min = df[TARGET].min()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["ds"], df[TARGET], color="steelblue", linewidth=1, label=TARGET)

ax.axvspan(train_df["ds"].iloc[0], train_df["ds"].iloc[-1],
           alpha=0.08, color="green", label="Training period")
ax.axvspan(test_df["ds"].iloc[0], test_df["ds"].iloc[-1],
           alpha=0.3, color="orange", label="Test window (forecast)")

for date, label in EVENTS.items():
    ts = pd.Timestamp(date)
    ax.axvline(ts, color="crimson", linestyle="--", linewidth=1, alpha=0.7)
    ax.text(ts, y_min + (y_max - y_min) * 0.05, label,
            rotation=90, fontsize=8, color="crimson", va="bottom")

ax.set_title("Monthly Employment — Historical Context with Economic Events")
ax.set_xlabel("Date")
ax.set_ylabel("Employment (thousands)")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small")

model = MoiraiForecast(
    prediction_length=12,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
    context_length=module.max_seq_len,
    module=module
)

model.eval()

print("past_length:", model.past_length)

In [ ]:
series = train_df[TARGET].values.astype(float)

required_len = model.past_length
series = series[-required_len:]

series = torch.tensor(series, dtype=torch.float32)\
    .unsqueeze(0)\
    .unsqueeze(-1)

print(series.shape)

In [ ]:
past_observed_target = torch.ones_like(series, dtype=torch.bool)

past_is_pad = torch.zeros(
    series.shape[0],
    series.shape[1],
    dtype=torch.bool
)

In [ ]:
with torch.no_grad():
    output = model(
        series,
        past_observed_target,
        past_is_pad
    )

print("Output shape:", output.shape)

In [ ]:
moirai_pred = output.mean(dim=1).squeeze(0).numpy()

print("Forecast shape:", moirai_pred.shape)
print("Forecast:", moirai_pred)

In [ ]:
moirai_mae = mean_absolute_error(y_true, moirai_pred)
moirai_mse = mean_squared_error(y_true, moirai_pred)
print(f"Moirai  MAE: {moirai_mae:.2f}")
print(f"Moirai  MSE: {moirai_mse:.2f}")

In [ ]:
forecast_dates = test_df["ds"].values

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(forecast_dates, y_true, color="black", marker="o", label="Actual", linewidth=2)
ax.plot(forecast_dates, moirai_pred, color="darkorange", label="Moirai", linewidth=1.5)
ax.set_title("Moirai Forecast vs Actual — Monthly Employment (2013\u20132014)")
ax.set_xlabel("Date")
ax.set_ylabel("Employment (thousands)")
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
moirai_std = output.std(dim=1).squeeze(0).numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(forecast_dates, y_true, color="black", marker="o", label="Actual", linewidth=2)
ax.plot(forecast_dates, moirai_pred, color="darkorange", label="Moirai", linewidth=1.5)
ax.fill_between(
    forecast_dates,
    moirai_pred - moirai_std,
    moirai_pred + moirai_std,
    alpha=0.25, color="darkorange", label="\u00b11 std"
)
ax.set_title("Moirai Forecast with Uncertainty — Monthly Employment")
ax.set_xlabel("Date")
ax.set_ylabel("Employment (thousands)")
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"\n{'Model':<12} {'MAE':>10} {'MSE':>14}")
print("-" * 38)
print(f"{'Moirai':<12} {moirai_mae:>10.2f} {moirai_mse:>14.2f}")